##Dataset

In [ ]:
!pip install datasets soundfile

In [ ]:
from datasets import load_dataset, concatenate_datasets, Dataset, Audio

def load_n_samples(name, label, n=400):
    stream = load_dataset("cgeorgiaw/animal-sounds", name, streaming=True)["train"]
    samples = []
    for i, example in enumerate(stream):
        if i >= n:
            break
        samples.append({
            "audio": example["audio"],
            "label": label
        })
    return Dataset.from_list(samples)

birds       = load_n_samples("birds",       "birds")
macaque     = load_n_samples("macaque",     "macaque")
giant_otter = load_n_samples("giant_otter", "giant_otter")
orca        = load_n_samples("orca",        "orca")
zebra_finch = load_n_samples("zebra_finch", "zebra_finch")

# Gabungkan
dataset = concatenate_datasets([birds, macaque, giant_otter, orca, zebra_finch])
print(f"Total sampel: {len(dataset)}")
print(f"Kolom      : {dataset.column_names}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/688 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/688 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/688 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/7286 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/688 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/595 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/688 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/3406 [00:00<?, ?it/s]

Total sampel: 2000
Kolom      : ['audio', 'label']


In [ ]:
print(dataset[0])    # sample dari birds
print(dataset[400])  # sample dari macaque
print(dataset[1600]) # sample dari zebra_finch

{'audio': {'bytes': None, 'path': 'hf://datasets/cgeorgiaw/animal-sounds@01bf9fefacf812e09e370e072c9f879c8964a62f/birds/nips4b_birds_trainfile001.wav'}, 'label': 'birds'}
{'audio': {'bytes': None, 'path': 'hf://datasets/cgeorgiaw/animal-sounds@01bf9fefacf812e09e370e072c9f879c8964a62f/macaque/TH28.wav'}, 'label': 'macaque'}
{'audio': {'bytes': None, 'path': 'hf://datasets/cgeorgiaw/animal-sounds@01bf9fefacf812e09e370e072c9f879c8964a62f/zebra_finch/WhiBlu4818_110404-DC-02.wav'}, 'label': 'zebra_finch'}


In [ ]:
print(f"Total sampel: {len(dataset)}")
print(f"Kolom      : {dataset.column_names}")

Total sampel: 2000
Kolom      : ['audio', 'label']


##Preprocessing


###1. Resampling

In [ ]:
from datasets import Audio

# Cast audio supaya bisa di-decode
dataset = dataset.cast_column("audio", Audio())

# Cek sample rate tiap kelas
for i, label in zip([0, 400, 800, 1200, 1600], ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]):
    sr = dataset[i]["audio"]["sampling_rate"]
    print(f"{label:20s} → sample rate: {sr} Hz")

birds                → sample rate: 44100 Hz
macaque              → sample rate: 24414 Hz
giant_otter          → sample rate: 96000 Hz
orca                 → sample rate: 44100 Hz
zebra_finch          → sample rate: 44100 Hz


In [ ]:
from datasets import Audio

# Resample semua ke 22050 Hz
dataset = dataset.cast_column("audio", Audio(sampling_rate=22050))

# Verifikasi
for i, label in zip([0, 400, 800, 1200, 1600], ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]):
    sr = dataset[i]["audio"]["sampling_rate"]
    print(f"{label:20s} → sample rate: {sr} Hz")

birds                → sample rate: 22050 Hz
macaque              → sample rate: 22050 Hz
giant_otter          → sample rate: 22050 Hz
orca                 → sample rate: 22050 Hz
zebra_finch          → sample rate: 22050 Hz


###2. Stereo ke Mono

In [ ]:
import numpy as np

for i, label in zip([0, 400, 800, 1200, 1600], ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]):
    audio = np.array(dataset[i]["audio"]["array"])
    print(f"{label:20s} → shape: {audio.shape}")

birds                → shape: (110336,)
macaque              → shape: (13573,)
giant_otter          → shape: (7714,)
orca                 → shape: (88200,)
zebra_finch          → shape: (4673,)


###3. Trim Silence

In [ ]:
import librosa
import numpy as np
import soundfile as sf
from io import BytesIO

def trim_silence(example):
    audio = np.array(example["audio"]["array"], dtype=np.float32)
    audio_trimmed, _ = librosa.effects.trim(audio, top_db=20)

    # Simpan sebagai bytes supaya tidak error saat encode
    buffer = BytesIO()
    sf.write(buffer, audio_trimmed, 22050, format="wav")
    example["audio"] = {"bytes": buffer.getvalue(), "path": None}
    return example

dataset = dataset.map(trim_silence)

# Verifikasi
from datasets import Audio
dataset = dataset.cast_column("audio", Audio(sampling_rate=22050))

for i, label in zip([0, 400, 800, 1200, 1600], ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]):
    audio = np.array(dataset[i]["audio"]["array"])
    duration = len(audio) / 22050
    print(f"{label:20s} → {len(audio)} sampel ({duration:.2f} detik)")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]